# 03 · What works? Drive style vs. outcomes

Three questions, in order of how much we can trust the answer:

1. **Do drive styles differ in outcomes once the situation is held fixed?** (adjusted, observational)
2. **Which play-call sequences go with touchdowns vs. punts?** (descriptive, length-controlled)
3. **How early can we tell how a drive will end, and does style add anything?** (out-of-time prediction)

The core problem is **confounding**: teams choose styles in particular situations (trailing, late in the half, bad field position),
and those situations drive outcomes too. Everything below is about not fooling ourselves.

Two definitions of 'style' are compared:
- **Whole-drive style** (Phase 3): contaminated by how the drive went (a stalled drive of incompletions *looks* hurry-up).
- **Opening style**: classified from only the first 3 designed plays, so it precedes the outcome. This is the cleaner treatment.

In [ ]:
import numpy as np, pandas as pd, polars as pl
import plotly.express as px, plotly.graph_objects as go
from drivedna.drives.build import DRIVE_KEY, PROCESSED_DIR
from drivedna.models.cluster import REPORTS_DIR, STYLE_ORDER
from drivedna.analysis.openings import OPENING_ORDER, build_opening_styles
from drivedna.analysis.modeling_table import build_model_table, OUTCOME_CLASSES

eff = pd.read_csv(REPORTS_DIR / "effectiveness.csv")
naive = pd.read_csv(REPORTS_DIR / "effectiveness_naive.csv")
opening_table, _ = build_opening_styles()
df = build_model_table().merge(opening_table.to_pandas(), on=DRIVE_KEY)
print(f"{len(df):,} eligible drives;", eff["spec"].nunique(), "specifications;", "bootstrap resamples whole games")
print(df["open_style"].value_counts(normalize=True).round(3).reindex(OPENING_ORDER).to_string())

## 1 · Teams choose styles in particular situations
If opening styles were assigned at random, these rows would look alike. They do differ, which is why a raw comparison misleads.

In [ ]:
sit = df.groupby("open_style")[["score_diff", "half_secs", "start_yardline_100", "off_strength", "def_strength"]].mean().reindex(OPENING_ORDER)
sit["pct_trailing"] = df.groupby("open_style").apply(lambda g: (g["score_diff"] < 0).mean()).reindex(OPENING_ORDER)
print(sit.round(2).to_string())
ov = pd.read_csv(REPORTS_DIR / "overlap_opening.csv", index_col=0).reindex(OPENING_ORDER)
print("\nOverlap: how well does the situation alone predict opening style? (low own-style propensity = style rarely used in comparable spots)")
print(ov.round(3).to_string())

## 2 · Main result: opening style, adjusted for the situation
Estimand **vs typical**: for drives that actually opened with style *s*, predicted outcome under *s* minus predicted outcome under the
share-weighted mix of styles in the same situation. Controls (all known before the first snap): field position, score, time left in the half,
pre-drive win probability, quarter, season, and the offense's and defense's strength (points/drive, shrunk, leave-one-game-out).
Bars are 95% intervals from bootstrapping whole games.

In [ ]:
def forest(spec, estimand, metric, order, title):
    d = eff.query("spec == @spec and estimand == @estimand and metric == @metric").set_index("style").reindex(order).reset_index()
    fig = go.Figure(go.Scatter(x=d["value"], y=d["style"], mode="markers", marker=dict(size=10),
        error_x=dict(type="data", symmetric=False, array=d["hi"] - d["value"], arrayminus=d["value"] - d["lo"])))
    fig.add_vline(x=0, line_dash="dot")
    fig.update_layout(title=title, height=330, xaxis_title=metric, yaxis=dict(autorange="reversed"))
    fig.show()
    return d[["style", "value", "lo", "hi"]]

forest("opening_style", "vs_typical", "points", OPENING_ORDER, "Points per drive vs. a typical opening in the same situation")
for m in ["scored", "Touchdown", "Punt", "Turnover", "epa"]:
    t = forest("opening_style", "vs_typical", m, OPENING_ORDER, f"{m}: opening style vs. typical (adjusted)")

In [ ]:
# How much did adjusting change the picture? (naive = raw difference from the league mean of the same metric)
n = naive.query("spec == 'opening_style'").set_index("style").reindex(OPENING_ORDER)
share = df["open_style"].value_counts(normalize=True).reindex(OPENING_ORDER)
adj = eff.query("spec == 'opening_style' and estimand == 'vs_typical'").pivot_table(index="style", columns="metric", values="value").reindex(OPENING_ORDER)
cmp = pd.DataFrame({
    "naive_points_vs_league": n["points"] - (n["points"] * share).sum(),
    "adjusted_points_vs_typical": adj["points"],
    "naive_scored_vs_league": n["scored"] - (n["scored"] * share).sum(),
    "adjusted_scored_vs_typical": adj["scored"],
})
print(cmp.round(3).to_string())

## 3 · Whole-drive style (for contrast; contaminated)
Same method, but style measured over the entire drive. This mixes 'what was called' with 'how the drive went', so treat it as descriptive only.

In [ ]:
forest("whole_drive_style", "vs_typical", "points", STYLE_ORDER, "Points per drive: WHOLE-drive style vs. typical (adjusted) - contaminated by how the drive went")

## 4 · Does the opening-style result survive sensitivity checks?
- **Team-season fixed effects:** compare a team with itself in the same season (soaks up QB, scheme, coach, roster).
- **Confident assignments only:** drives in the top half of opening-style assignment margin.
- **Subgroups:** trailing vs. leading/tied; backed up inside the own 20 vs. not.

In [ ]:
specs = ["opening_style", "opening_team_season_fe", "opening_confident_half", "opening_trailing", "opening_leading_or_tied", "opening_backed_up", "opening_not_backed_up"]
d = eff.query("spec in @specs and estimand == 'vs_typical' and metric == 'points'").copy()
d["label"] = d.apply(lambda r: f"{r['value']:+.2f} [{r['lo']:+.2f}, {r['hi']:+.2f}]", axis=1)
print(d.pivot(index="style", columns="spec", values="label").reindex(OPENING_ORDER)[specs].to_string())
print("\ndrives per spec:", eff.drop_duplicates("spec").set_index("spec")["n_drives"].reindex(specs).to_dict())
fig = px.scatter(d, x="value", y="style", color="spec", error_x=d["hi"] - d["value"], error_x_minus=d["value"] - d["lo"],
                 category_orders={"style": OPENING_ORDER}, height=520, title="Points per drive vs. typical, by specification")
fig.add_vline(x=0, line_dash="dot"); fig.update_yaxes(autorange="reversed"); fig.show()

## 5 · Which call sequences go with touchdowns vs. punts?
Touchdown drives are longer than punt drives, so almost any n-gram looks 'touchdown-leaning' just because there are more plays.
Each n-gram is tested **within drive-length strata** (Mantel-Haenszel odds ratio), with Benjamini-Hochberg FDR across all tests.
*Sacks are a result, not a call, so n-grams containing `SACK` are shown separately.*

In [ ]:
sa = pd.read_csv(REPORTS_DIR / "sequences_anywhere.csv")
print(f"{len(sa)} n-grams tested; {int(sa.significant.sum())} significant at FDR 5%; "
      f"naive OR > 1.2 for {(sa.or_naive > 1.2).mean():.0%} of them vs. stratified OR > 1.2 & significant for {((sa.or_stratified > 1.2) & sa.significant).mean():.0%}")
fig = px.scatter(sa, x=np.log(sa["or_naive"]), y=np.log(sa["or_stratified"]), hover_name="ngram", opacity=0.6,
                 labels={"x": "log odds ratio, naive", "y": "log odds ratio, length-stratified"},
                 title="Stratifying by drive length shrinks almost every 'touchdown pattern' (points below the diagonal)")
fig.add_shape(type="line", x0=-4, y0=-4, x1=4, y1=4, line_dash="dot"); fig.show()

In [ ]:
cols = ["ngram", "drives", "share_Touchdown", "share_Punt", "or_stratified", "or_lo", "or_hi", "q"]
nosack = sa[~sa["ngram"].str.contains("SACK")]
print("Touchdown-leaning calls (anywhere in the drive, length-controlled):"); print(nosack.head(10)[cols].round(3).to_string(index=False))
print("\nPunt-leaning calls (excluding sacks):"); print(nosack.tail(10)[cols].round(3).to_string(index=False))
print("\nStrongest punt markers are sacks (a result): "); print(sa[sa["ngram"].str.contains("SACK")].tail(4)[cols].round(3).to_string(index=False))

In [ ]:
so = pd.read_csv(REPORTS_DIR / "sequences_opening.csv")
print(f"OPENING view (first 4 calls, drives lasting >= 6 plays): {len(so)} n-grams, {int(so.significant.sum())} significant")
print(so.head(8)[cols].round(3).to_string(index=False)); print(so.tail(8)[cols].round(3).to_string(index=False))

## 6 · How early can we tell? Out-of-time prediction after 3 plays
Train on 2016-2022, test on 2023-2025. Team strength uses prior weeks only (no future games). Drives that end within 3 plays are excluded.
**progress** = what happened in the 3 plays and the down/distance/field position facing the 4th.

In [ ]:
ep = pd.read_csv(REPORTS_DIR / "early_prediction.csv")
for target in ["scored", "touchdown"]:
    m = ep.query("target == @target and kind != 'increment'")[["model", "kind", "log_loss", "brier", "auc"]]
    print(f"--- target: {target}"); print(m.round(4).to_string(index=False))
    inc = ep.query("target == @target and kind == 'increment'")[["model", "improvement", "lo", "hi"]]
    print("log-loss improvement (positive = better):"); print(inc.round(4).to_string(index=False))

In [ ]:
imp = pd.read_csv(REPORTS_DIR / "early_importance.csv").query("target == 'scored'")
grp = imp[imp["is_group"]].sort_values("delta_log_loss")
px.bar(grp, x="delta_log_loss", y="feature_or_group", orientation="h", title="Permutation importance of feature groups (increase in test log-loss when shuffled)").show()
print(imp[~imp["is_group"]].head(8)[["feature_or_group", "delta_log_loss"]].round(4).to_string(index=False))

## Takeaways

**1. The way a drive opens is associated with how it ends, holding the situation fixed.**
Compared with a typical opening in the same situation (roughly a sixth of the ~2 points a drive is worth):

| Opening (first 3 plays) | Points / drive | Share of openings |
|---|---|---|
| **Deep-shot** (82% of passes deep) | **+0.36** [+0.29, +0.42] | 14% |
| Run-first (25% passing) | +0.11 [+0.06, +0.15] | 26% |
| Balanced, inside or outside runs | -0.12 to -0.14 | 23% / 20% |
| Pass-only, up-tempo | -0.13 [-0.18, -0.07] | 19% |

Deep shots also add ~5.6 points of scoring probability (touchdown or field goal) and ~0.38 EPA per drive. The inside-vs-outside run split among balanced openers carries no outcome information.

**2. The deep-shot result is robust.** It stays between +0.29 and +0.54 in every specification: comparing each team with itself within a season (+0.31, which soaks up QB, scheme and roster), confident assignments only (+0.54), trailing (+0.36), leading or tied (+0.35), backed up inside the own 20 (+0.29), and not backed up (+0.38). Adjusting for field position, score, time, win probability and team strength barely moves any estimate (e.g. raw +0.34 vs adjusted +0.36), so measured context explains little of the gap.

**3. One real exception.** The Pass-only up-tempo penalty (-0.13 overall, -0.18 when not backed up) disappears when the offense starts inside its own 20 (+0.01 [-0.10, +0.13]).

**4. Why: it runs through progress, not style itself.** After the first 3 plays, predicting whether the drive scores (test 2023-25, AUC 0.73 vs 0.63 from context alone) depends overwhelmingly on where the drive stands, above all the field position facing the 4th play. Opening style adds a little before the plays are known (+0.005 log-loss, CI +0.003 to +0.008) and **nothing once progress is known** (slightly negative). That fits "deep shots help by gaining yards early," but this is consistent with the mechanism, not a mediation analysis that proves it. Gradient boosting does no better than logistic regression.

**5. Call sequences.** Naively, 92% of 350 n-grams look touchdown-leaning, purely because touchdown drives are longer; comparing within drive-length strata leaves 24% significant and touchdown-leaning, with smaller odds ratios. The touchdown patterns all contain a deep ball (odds ratio ~2.2-2.9), and in the opening window `P_deep` early again gives ~1.9-2.6. Punt-leaning patterns are strings of runs and short passes (e.g. `R_in P_short R_in` at 0.50). The strongest punt markers are sacks (odds ratios 0.01-0.09), but a sack is a result, not a call.

**Caveats, in order of importance**
- **Observational.** We adjust for what is measured, not for what a coach sees (coverage, personnel, the QB's arm in a specific moment). Team-season fixed effects help but cannot rule this out. Treat these as strong associations, not proven causal effects.
- **"Better than a typical opening" is not "optimal."** Deep shots carry risk and may not be callable on every down; the data shows how the openings teams actually chose fared.
- **Whole-drive style is contaminated** (section 3 of this notebook): it gives No-huddle +0.49 and Hurry-up -0.40, partly because a stalled drive looks hurry-up. Use opening style for any claim about effectiveness.
- **Drives that end within 3 designed plays are excluded** from the early-prediction task (no 4th play to condition on), and drives under 3 plays from everything.
